<a href="https://colab.research.google.com/github/Lanzero225/arXiv-Project/blob/main/ReZearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
torch.cuda.is_available()

False

In [ ]:
import pandas as pd

dataset_file = "/kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json"

chunksize = 100_000
chunks = []

def preprocess_chunk(chunk):
    chunk['text'] = chunk['title'] + ". " + chunk['abstract']
    return chunk[['id', 'text', 'categories']]

processed_chunks = []
for chunk in pd.read_json(dataset_file, lines=True, chunksize=100_000):
    chunk = preprocess_chunk(chunk)
    processed_chunks.append(chunk)

dataset = pd.concat(processed_chunks, ignore_index=True)

print(dataset.shape)

(2963163, 3)


In [ ]:
dataset.head()

,id,title,authors,journal-ref,doi,categories,abstract
0,704.0001,Calculation of prompt diphoton production cros...,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,hep-ph,A fully differential calculation in perturba...
1,704.0002,Sparsity-certifying Graph Decompositions,Ileana Streinu and Louis Theran,None,None,math.CO cs.CG,"We describe a new algorithm, the $(k,\ell)$-..."
2,704.0003,The evolution of the Earth-Moon system based o...,Hongjun Pan,None,None,physics.gen-ph,The evolution of Earth-Moon system is descri...
3,704.0004,A determinant of Stirling cycle numbers counts...,David Callan,None,None,math.CO,We show that a determinant of Stirling cycle...
4,704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,Wael Abu-Shammala and Alberto Torchinsky,"Illinois J. Math. 52 (2008) no.2, 681-689",None,math.CA math.FA,In this paper we show how to compute the $\L...


In [ ]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2963163 entries, 0 to 2963162
Data columns (total 7 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   id           object
 1   title        object
 2   authors      object
 3   journal-ref  object
 4   doi          object
 5   categories   object
 6   abstract     object
dtypes: object(7)
memory usage: 158.3+ MB


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

sample_dataset = dataset.sample(2000, random_state=42)
embeddings = model.encode(sample_dataset['text'].tolist(), batch_size=64, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 31.0 MB/s eta 0:00:00


In [ ]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings, dtype='float32'))

In [ ]:
query = "deep learning medical imaging"
query_vec = model.encode([query])

In [ ]:
index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x785b30874690> >

In [ ]:
k = 10
distances, indices = index.search(np.array(query_vec), k)

results = dataset.iloc[indices[0]]
results[['text', 'categories']]

,text,categories
273,"New version announcement for TaylUR, an arbitr...",physics.comp-ph
1508,Modeling the Solar Chromosphere. Spectral di...,astro-ph
1096,Zero-temperature resistive transition in Josep...,cond-mat.supr-con cond-mat.stat-mech
1954,Metrical characterization of super-reflexivity...,math.FA math.MG
923,Lower order terms in the 1-level density for f...,math.NT
1610,Burgers Turbulence. The last decades witness...,nlin.CD
868,Connected Operators for the Totally Asymmetric...,cond-mat.stat-mech math-ph math.MP
1585,Core excitation in the elastic scattering and ...,nucl-th
1475,TeV-scale gravity in Horava-Witten theory on a...,hep-th
84,A Universality in PP-Waves. We discuss a uni...,hep-th


In [ ]:
cs_dataset = sample_dataset[sample_dataset['categories'].str.startswith('cs.')].reset_index(drop=True)
cs_embeddings = embeddings[cs_dataset.index]

dimension = cs_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(cs_embeddings, dtype='float32'))

query_vec = model.encode([query])
k = 5
distances, indices = index.search(np.array(query_vec, dtype='float32'), k)

results = cs_dataset.iloc[indices[0]]
print(results[['text', 'categories']])

                                                  text         categories
13   Enhancing Maritime Situational Awareness throu...              cs.CV
63   Efficient and Secure TSA for the Tangle.   The...  cs.CR cs.CC cs.NI
150  Transformers for One-Shot Visual Imitation.   ...  cs.LG cs.CV cs.RO
58   Gate Recurrent Unit for Efficient Industrial G...        cs.LG cs.AI
33   A Generic Coordinate Descent Framework for Lea...        cs.IR cs.LG
